# INVEST - 부동산담보대출 투자적격 심사 분석 모델
## 구성
- **Model 1**: 투자적격판단모델 (Classification) → target: `ivt_jg_cm_cd`
- **Model 2**: 적정금리평가모델 (Regression) → target: `ln_itt`
- **데이터**: Athena `mlops.altinv_crel_train`
- **MLFlow**: Tracking + Model Registry 연동
- **출력**: 4개 차트 (모델결과 2 + 변수중요도 2)

---
## 1부: TRAIN (모델 학습 및 MLFlow 등록)

### 1-1. 환경 설정 및 라이브러리

In [ ]:
import os
import tempfile
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
import xgboost as xgb

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import awswrangler as wr
import boto3

# ── 한글 폰트 설정: NanumGothic.ttf 직접 로드
import matplotlib.font_manager as fm
from pathlib import Path

# ttf 파일 경로 (노트북과 같은 디렉토리 기준, 필요시 절대경로로 변경)
FONT_PATH = Path('./NanumGothic.ttf')

if FONT_PATH.exists():
    fm.fontManager.addfont(str(FONT_PATH))
    prop = fm.FontProperties(fname=str(FONT_PATH))
    plt.rcParams['font.family'] = prop.get_name()
    print(f'폰트 로드 성공: {prop.get_name()}')
else:
    print(f'[Warning] {FONT_PATH} 없음 → DejaVu Sans 사용')
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False

print('Libraries loaded OK')

### 1-2. MLFlow 설정

In [ ]:
from dotenv import load_dotenv
import os

# 노트북과 같은 디렉토리의 .env 파일 로드
load_dotenv(dotenv_path='.env', override=True)

# 환경변수 확인 (값은 출력 안 함)
required = ['MLFLOW_TRACKING_URI', 'MLFLOW_TRACKING_USERNAME', 'MLFLOW_TRACKING_PASSWORD', 'AWS_REGION']
for key in required:
    val = os.getenv(key)
    print(f'{key}: {"OK" if val else "NOT SET"}')

MLFLOW_URI    = os.environ['MLFLOW_TRACKING_URI']
ATHENA_DB     = os.getenv('ATHENA_DB', 'mlops')
S3_OUTPUT     = os.getenv('ATHENA_S3_OUTPUT', 's3://s3-an2-mlops/athena/')

mlflow.set_tracking_uri(MLFLOW_URI)
EXPERIMENT_NAME = 'invest-crel-model'
mlflow.set_experiment(EXPERIMENT_NAME)

print('\nMLFlow URI:', mlflow.get_tracking_uri())
print('Experiment:', EXPERIMENT_NAME)

### 1-3. Athena 데이터 로딩

In [ ]:
ATHENA_DB    = os.getenv('ATHENA_DB', 'mlops')
ATHENA_TABLE = 'altinv_crel_train'
S3_OUTPUT    = os.getenv('ATHENA_S3_OUTPUT', 's3://s3-an2-mlops/athena/')
S3_BUCKET    = os.getenv('S3_BUCKET', 's3-an2-mlops')

query = f'SELECT * FROM {ATHENA_DB}.{ATHENA_TABLE}'

df_raw = wr.athena.read_sql_query(
    sql=query,
    database=ATHENA_DB,
    s3_output=S3_OUTPUT
)

print(f'ATHENA_DB   : {ATHENA_DB}')
print(f'ATHENA_TABLE: {ATHENA_TABLE}')
print(f'S3_OUTPUT   : {S3_OUTPUT}')
print(f'Shape: {df_raw.shape}')
df_raw.head(3)


### 1-4. 전처리 및 피처 엔지니어링

In [ ]:
df = df_raw.copy()

# ── 컬럼 정의
NUMERIC_COLS = [
    'gpt_ivt_trc_pi_rk', 'gpt_ivt_dlb_rqt_amt', 'ln_pd',
    'ltv_rte', 'gpt_ivt_cpt_ern_rte', 'gpt_ivt_cpt_ern_pd',
    'gpt_ivt_all_pcm_amt', 'gpt_ivt_bdg_scl_txt', 'gpt_ivt_nwk_ot_scl_txt',
    'gpt_ivt_cmpi_yr', 'dbt_rpy_coef_rte', 'gpt_ivt_rmd_lsg_ycn',
    'gpt_ivt_etrm_rte', 'gpt_ivt_mkt_avg_etrm_rt',
    'gpt_ivt_ppo_re_amt', 'gpt_ivt_mkt_ppo_re_amt',
    'gpt_ivt_mkt_avg_cpt_rte', 'gpt_ivt_mkt_avg_dln_amt',
    'gpt_ivt_ln_pfat_txt', 'gpt_ivt_te_ppo_amt',
    'gpt_ivt_cpt_reim', 'gpt_ivt_appr_evl_ppo_amt',
    'gpt_ivt_rpy_rte', 'bs_itt'
]

CAT_COLS = [
    'gpt_ivt_mth_cd', 'gpt_ivt_ser_dv_cd', 'gpt_ivt_tp_cd',
    'gpt_ivt_str_dv_cd', 'gpt_ivt_kd_cd', 'gpt_ivt_ara_dv_cd',
    'gpt_ivt_crd_rinf_txt', 'gpt_ivt_ecfr_gd_txt'
]

TARGET_CLS = 'ivt_jg_cm_cd'   # 투자적격판단 (분류)
TARGET_REG = 'ln_itt'          # 적정금리 (회귀)

# ── 숫자형 변환 및 결측 처리
for col in NUMERIC_COLS + [TARGET_REG]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[NUMERIC_COLS] = df[NUMERIC_COLS].fillna(df[NUMERIC_COLS].median())

# ── 범주형 처리 (Label Encoding)
le_dict = {}
for col in CAT_COLS:
    df[col] = df[col].fillna('UNKNOWN').astype(str)
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_dict[col] = le

# ── 분류 타겟 처리
df[TARGET_CLS] = df[TARGET_CLS].fillna('N').astype(str)
le_target = LabelEncoder()
df[TARGET_CLS] = le_target.fit_transform(df[TARGET_CLS])
print('Target classes:', le_target.classes_)

# ── 회귀 타겟 결측 제거
df = df.dropna(subset=[TARGET_REG])

FEATURE_COLS = NUMERIC_COLS + CAT_COLS
print(f'Feature count: {len(FEATURE_COLS)}, Rows: {len(df)}')
df[FEATURE_COLS + [TARGET_CLS, TARGET_REG]].describe()

---
### 1-5. Model 1 - 투자적격판단모델 (XGBoost Classification)

In [ ]:
X = df[FEATURE_COLS]
y_cls = df[TARGET_CLS]

class_counts = y_cls.value_counts()
print('클래스 분포:')
print(class_counts)

n = len(df)
min_class_count = class_counts.min()
use_split = (n >= 20) and (min_class_count >= 2)

if use_split:
    use_stratify = y_cls if min_class_count >= 2 else None
    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
        X, y_cls, test_size=0.2, random_state=42, stratify=use_stratify
    )
    eval_set = [(X_test_c, y_test_c)] if set(y_train_c.unique()) == set(y_test_c.unique()) else None
else:
    print(f'[Warning] 데이터 {n}건 → 전체 데이터로 학습')
    X_train_c, y_train_c = X, y_cls
    X_test_c,  y_test_c  = X, y_cls
    eval_set = None

print(f'Train: {len(X_train_c)}, Test: {len(X_test_c)}')

CLS_PARAMS = {
    'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'eval_metric': 'logloss', 'random_state': 42
}


with mlflow.start_run(run_name='crel-invest-classification') as run_cls:
    clf = xgb.XGBClassifier(**CLS_PARAMS)
    clf.fit(X_train_c, y_train_c, eval_set=eval_set, verbose=False)

    y_pred_c = clf.predict(X_test_c)
    y_prob_c = clf.predict_proba(X_test_c)
    n_classes = len(np.unique(y_cls))
    acc = (y_pred_c == y_test_c).mean()
    try:
        auc = roc_auc_score(y_test_c, y_prob_c[:, 1]) if n_classes == 2 \
              else roc_auc_score(y_test_c, y_prob_c, multi_class='ovr', average='macro')
    except ValueError as e:
        print(f'[Warning] ROC-AUC 계산 불가: {e}')
        auc = 0.0

    mlflow.log_params(CLS_PARAMS)
    mlflow.log_param('model_type', 'XGBClassifier')
    mlflow.log_param('target', TARGET_CLS)
    mlflow.log_param('feature_count', len(FEATURE_COLS))
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('roc_auc', auc)
    mlflow.log_metric('train_rows', len(X_train_c))
    mlflow.log_metric('test_rows', len(X_test_c))

    # log_model 대신 save_model → log_artifacts 우회 (MLflow 서버 중복키 버그 회피)
    with tempfile.TemporaryDirectory() as tmp:
        mlflow.xgboost.save_model(clf, os.path.join(tmp, 'model'))
        mlflow.log_artifacts(os.path.join(tmp, 'model'), artifact_path='model')

    RUN_ID_CLS = run_cls.info.run_id

# run 종료 후 Registry 등록
mlflow.register_model(f'runs:/{RUN_ID_CLS}/model', 'invest-crel-classification')

print(f'run_id: {RUN_ID_CLS}')
print(f'Accuracy: {acc:.4f}  |  ROC-AUC: {auc:.4f}')
print(classification_report(y_test_c, y_pred_c, target_names=le_target.classes_))


### 1-6. Model 2 - 적정금리평가모델 (XGBoost Regression)

In [ ]:
y_reg = df[TARGET_REG]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train_r)}, Test: {len(X_test_r)}')

REG_PARAMS = {
    'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42
}

with mlflow.start_run(run_name='crel-invest-regression') as run_reg:
    reg = xgb.XGBRegressor(**REG_PARAMS)
    reg.fit(X_train_r, y_train_r,
            eval_set=[(X_test_r, y_test_r)], verbose=False)

    y_pred_r = reg.predict(X_test_r)
    mae  = mean_absolute_error(y_test_r, y_pred_r)
    rmse = mean_squared_error(y_test_r, y_pred_r) ** 0.5
    r2   = r2_score(y_test_r, y_pred_r)

    mlflow.log_params(REG_PARAMS)
    mlflow.log_param('model_type', 'XGBRegressor')
    mlflow.log_param('target', TARGET_REG)
    mlflow.log_param('feature_count', len(FEATURE_COLS))
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('r2', r2)
    mlflow.log_metric('train_rows', len(X_train_r))
    mlflow.log_metric('test_rows', len(X_test_r))

    # log_model 대신 save_model → log_artifacts 우회
    with tempfile.TemporaryDirectory() as tmp:
        mlflow.xgboost.save_model(reg, os.path.join(tmp, 'model'))
        mlflow.log_artifacts(os.path.join(tmp, 'model'), artifact_path='model')

    RUN_ID_REG = run_reg.info.run_id

# run 종료 후 Registry 등록
mlflow.register_model(f'runs:/{RUN_ID_REG}/model', 'invest-crel-regression')

print(f'run_id: {RUN_ID_REG}')
print(f'MAE: {mae:.4f}  |  RMSE: {rmse:.4f}  |  R²: {r2:.4f}')


In [ ]:
# ── LabelEncoder & 전처리 통계 MLFlow Artifact 등록
# API 서버(inference.py)에서 동일 encoder를 로드하기 위해 필수
import os, joblib, tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    enc_dir = os.path.join(tmpdir, 'encoders')
    os.makedirs(enc_dir)

    # LabelEncoder 저장
    joblib.dump(le_dict,   os.path.join(enc_dir, 'le_dict.pkl'))
    joblib.dump(le_target, os.path.join(enc_dir, 'le_target.pkl'))

    # 수치형 컬럼 중앙값 저장 (API 단건 추론 시 결측 대체용)
    import json as _json
    medians = df[NUMERIC_COLS].median().to_dict()
    with open(os.path.join(enc_dir, 'numeric_medians.json'), 'w') as f:
        _json.dump(medians, f, ensure_ascii=False, indent=2)

    # 피처 컬럼 목록 저장
    with open(os.path.join(enc_dir, 'feature_cols.json'), 'w') as f:
        _json.dump({'numeric_cols': NUMERIC_COLS, 'cat_cols': CAT_COLS,
                    'feature_cols': FEATURE_COLS}, f)

    # 두 run에 모두 등록
    for run_id in [RUN_ID_CLS, RUN_ID_REG]:
        with mlflow.start_run(run_id=run_id):
            mlflow.log_artifacts(enc_dir, artifact_path='encoders')

    saved = os.listdir(enc_dir)
    print('Artifact 등록 완료:')
    for f in saved:
        print(f'  encoders/{f}')
    print(f'\nRUN_ID_CLS: {RUN_ID_CLS}')
    print(f'RUN_ID_REG: {RUN_ID_REG}')
    print('\nMLFlow UI → Runs > Artifacts > encoders/ 에서 확인 가능')


### 1-7. 결과 시각화 (4개 차트)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('INVEST 부동산담보대출 심사모델 분석 결과', fontsize=16, fontweight='bold')

# ── Chart 1: 투자적격판단 ROC Curve
ax1 = axes[0, 0]
if n_classes == 2:
    fpr, tpr, _ = roc_curve(y_test_c, y_prob_c[:, 1])
    ax1.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC={auc:.3f})')
    ax1.plot([0,1],[0,1], 'k--', lw=1)
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
else:
    ax1.bar(le_target.classes_, np.bincount(y_test_c) / len(y_test_c), color='steelblue')
    ax1.set_ylabel('비율')
ax1.set_title(f'[Model 1] 투자적격판단 - ROC Curve (AUC={auc:.3f})')
ax1.legend(loc='lower right')
ax1.grid(alpha=0.3)

# ── Chart 2: 적정금리 실제 vs 예측
ax2 = axes[0, 1]
ax2.scatter(y_test_r, y_pred_r, alpha=0.5, color='coral', edgecolors='none', s=30)
lims = [min(y_test_r.min(), y_pred_r.min()), max(y_test_r.max(), y_pred_r.max())]
ax2.plot(lims, lims, 'k--', lw=1)
ax2.set_xlabel('실제 금리 (ln_itt)')
ax2.set_ylabel('예측 금리')
ax2.set_title(f'[Model 2] 적정금리평가 - 실제 vs 예측\nR²={r2:.3f}, RMSE={rmse:.3f}')
ax2.grid(alpha=0.3)

# ── Chart 3: Model 1 변수 중요도 (Top 15)
ax3 = axes[1, 0]
fi_cls = pd.Series(clf.feature_importances_, index=FEATURE_COLS).nlargest(15)
fi_cls.sort_values().plot(kind='barh', ax=ax3, color='steelblue', alpha=0.8)
ax3.set_title('[Model 1] 투자적격판단 - 변수 중요도 Top 15')
ax3.set_xlabel('Feature Importance')
ax3.grid(axis='x', alpha=0.3)

# ── Chart 4: Model 2 변수 중요도 (Top 15)
ax4 = axes[1, 1]
fi_reg = pd.Series(reg.feature_importances_, index=FEATURE_COLS).nlargest(15)
fi_reg.sort_values().plot(kind='barh', ax=ax4, color='coral', alpha=0.8)
ax4.set_title('[Model 2] 적정금리평가 - 변수 중요도 Top 15')
ax4.set_xlabel('Feature Importance')
ax4.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('invest_crel_results.png', dpi=150, bbox_inches='tight')

with mlflow.start_run(run_id=RUN_ID_CLS):
    mlflow.log_artifact('invest_crel_results.png')

plt.show()
print('차트 저장 완료: invest_crel_results.png')


---
## 2부: OPS (추론)
MLFlow Registry에서 모델을 로드하여 단건 또는 배치 예측 수행

### 2-1. MLFlow Registry에서 모델 로드

In [ ]:
import mlflow.pyfunc

# 최신 Production 버전 로드
# 버전 고정 시: 'models:/invest-crel-classification/1'
MODEL_CLS_URI = 'models:/invest-crel-classification/latest'
MODEL_REG_URI = 'models:/invest-crel-regression/latest'

loaded_clf = mlflow.pyfunc.load_model(MODEL_CLS_URI)
loaded_reg = mlflow.pyfunc.load_model(MODEL_REG_URI)

print('투자적격판단 모델 로드 완료:', MODEL_CLS_URI)
print('적정금리평가 모델 로드 완료:', MODEL_REG_URI)

### 2-2. 단건 추론 예시

In [ ]:
# 추론용 샘플 입력 (실제 운영 시 PDF 파싱 결과를 여기에 매핑)
sample_input = pd.DataFrame([{
    'gpt_ivt_trc_pi_rk':       1.0,
    'gpt_ivt_dlb_rqt_amt':     50000000000.0,  # 500억
    'ln_pd':                   24.0,            # 24개월
    'ltv_rte':                 65.0,            # LTV 65%
    'gpt_ivt_cpt_ern_rte':     8.5,
    'gpt_ivt_cpt_ern_pd':      36.0,
    'gpt_ivt_all_pcm_amt':     80000000000.0,
    'gpt_ivt_bdg_scl_txt':     15000.0,        # 평
    'gpt_ivt_nwk_ot_scl_txt':  2000000000000.0,
    'gpt_ivt_cmpi_yr':         2018.0,
    'dbt_rpy_coef_rte':        1.35,            # DSCR
    'gpt_ivt_rmd_lsg_ycn':     3.5,
    'gpt_ivt_etrm_rte':        5.0,
    'gpt_ivt_mkt_avg_etrm_rt': 7.0,
    'gpt_ivt_ppo_re_amt':      45000.0,
    'gpt_ivt_mkt_ppo_re_amt':  43000.0,
    'gpt_ivt_mkt_avg_cpt_rte': 4.2,
    'gpt_ivt_mkt_avg_dln_amt': 2800.0,
    'gpt_ivt_ln_pfat_txt':     8.0,            # Debt Yield
    'gpt_ivt_te_ppo_amt':      3000.0,
    'gpt_ivt_cpt_reim':        30.0,
    'gpt_ivt_appr_evl_ppo_amt':2900.0,
    'gpt_ivt_rpy_rte':         0.0,
    'bs_itt':                  3.5,             # 기준금리
    # 범주형 (학습 시 LabelEncoding된 값으로 변환 필요)
    'gpt_ivt_mth_cd':          0,
    'gpt_ivt_ser_dv_cd':       1,
    'gpt_ivt_tp_cd':           0,
    'gpt_ivt_str_dv_cd':       0,
    'gpt_ivt_kd_cd':           2,
    'gpt_ivt_ara_dv_cd':       0,
    'gpt_ivt_crd_rinf_txt':    1,
    'gpt_ivt_ecfr_gd_txt':     0,
}])

pred_cls  = loaded_clf.predict(sample_input)
pred_rate = loaded_reg.predict(sample_input)

cls_label = le_target.inverse_transform(pred_cls)

print('=' * 40)
print('[ 투자적격 심사 결과 ]')
print(f'  심사승인여부: {cls_label[0]}')
print(f'  적정 수익률:  {pred_rate[0]:.4f} (%)')
print('=' * 40)

### 2-3. 배치 추론 (S3 CSV → 예측 → S3 저장)

In [ ]:
# 배치 데이터 로드 - Athena 조회 (datagen 노트북으로 적재한 parquet 기준)
S3_BATCH_OUTPUT = f's3://{S3_BUCKET}/aimodel/altinv_crel_train/crel_inference_result.csv'

print(f'데이터 로드: {ATHENA_DB}.{ATHENA_TABLE}')
df_batch = wr.athena.read_sql_query(
    sql=f'SELECT * FROM {ATHENA_DB}.{ATHENA_TABLE}',
    database=ATHENA_DB,
    s3_output=S3_OUTPUT
)
print(f'배치 입력: {df_batch.shape}')

# 전처리 (학습과 동일 로직)
df_batch_proc = df_batch.copy()
for col in NUMERIC_COLS:
    df_batch_proc[col] = pd.to_numeric(df_batch_proc[col], errors='coerce')
df_batch_proc[NUMERIC_COLS] = df_batch_proc[NUMERIC_COLS].fillna(df_batch_proc[NUMERIC_COLS].median())

for col in CAT_COLS:
    df_batch_proc[col] = df_batch_proc[col].fillna('UNKNOWN').astype(str)
    known = set(le_dict[col].classes_)
    df_batch_proc[col] = df_batch_proc[col].apply(
        lambda x: x if x in known else le_dict[col].classes_[0]
    )
    df_batch_proc[col] = le_dict[col].transform(df_batch_proc[col])

X_batch = df_batch_proc[FEATURE_COLS]

# 예측
batch_cls  = loaded_clf.predict(X_batch)
batch_rate = loaded_reg.predict(X_batch)

df_batch['pred_invest_yn'] = le_target.inverse_transform(batch_cls)
df_batch['pred_fair_rate'] = batch_rate

# 결과 S3 저장
wr.s3.to_csv(df_batch, path=S3_BATCH_OUTPUT, index=False)
print(f'배치 추론 완료 → {S3_BATCH_OUTPUT}')
df_batch[['gpt_ivt_jg_seq', 'pred_invest_yn', 'pred_fair_rate']].head(10)


### 2-4. MLFlow Model Serving API 호출 (Optional)

In [ ]:
import requests, json

SERVING_URL_CLS = 'http://mlflow.mlflow.svc.cluster.local:80/invocations'

mlflow_user = os.environ.get('MLFLOW_TRACKING_USERNAME')
mlflow_pass = os.environ.get('MLFLOW_TRACKING_PASSWORD')

if not mlflow_user or not mlflow_pass:
    raise EnvironmentError('.env에 MLFLOW_TRACKING_USERNAME / MLFLOW_TRACKING_PASSWORD 설정 필요')

headers = {'Content-Type': 'application/json'}
auth    = (mlflow_user, mlflow_pass)
payload = {'dataframe_records': sample_input.to_dict(orient='records')}

try:
    resp = requests.post(
        SERVING_URL_CLS, headers=headers,
        data=json.dumps(payload), auth=auth, timeout=30
    )
    print('Status:', resp.status_code)
    if resp.status_code == 200:
        print('Response:', resp.json())
    elif resp.status_code == 401:
        print('[인증 오류] USERNAME/PASSWORD 확인 필요')
    elif resp.status_code == 404:
        print('[404] 모델 서빙 엔드포인트 미배포 상태')
    else:
        print('응답 내용:', resp.text[:500])
except requests.exceptions.ConnectionError:
    print('[연결 실패] Serving 미기동 → 로컬 모델 사용')
except Exception as e:
    print(f'오류: {e}')


### 2-5. MLFlow 모델 버전 관리 (Registry 조회)

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

for model_name in ['invest-crel-classification', 'invest-crel-regression']:
    print(f'\n[ {model_name} ]')
    versions = client.search_model_versions(f"name='{model_name}'")
    for v in versions:
        print(f'  version={v.version}, stage={v.current_stage}, run_id={v.run_id[:8]}...')